#  pytorch로 CNN 모델링하기

## 1.환경준비

### (1) 라이브러리 로딩

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchsummary import summary

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

### (2) 데이터로딩

In [ ]:
train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=ToTensor() )
test_dataset = datasets.MNIST(root="data", train=False, download=True, transform=ToTensor() )

## 2.데이터 준비

### (1) 학습 데이터 로더

In [ ]:
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for x, y in train_loader:
    print(f"Shape of X [N, C, H, W]: {x.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

### (2) 검증/테스트 데이터셋

In [ ]:
# test_dataset을 5000건씩 val : test로 분할
x_val, x_test = test_dataset.data[:5000], test_dataset.data[5000:]
y_val, y_test = test_dataset.targets[:5000], test_dataset.targets[5000:]

# 스케일링
x_val = x_val / 255
x_test = x_test/ 255

# 차원 맞추기
print(x_val.shape, x_test.shape)
x_val = x_val.view(5000, 1, 28, 28)
x_test = x_test.view(5000, 1, 28, 28)
print(x_val.shape, x_test.shape)

## 3.CNN 모델링1

### (1) 모델 설계

In [ ]:
model = nn.Sequential(
                    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2, stride=2),
                    nn.Flatten(),
                    nn.Linear(32*14*14, 10)
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

In [ ]:
summary(model, input_size=(1, 28, 28))

### (2) 학습

#### 1) 필요 함수 생성

* 학습 함수

In [ ]:
def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)                  # 전체 데이터셋의 크기
    num_batches = len(dataloader)                   # 배치 크기
    tr_loss = 0
    model.train()                                   # 훈련 모드로 설정
    for x, y in dataloader:     # batch 단위 로딩
        x, y = x.to(device), y.to(device)      # 디바이스 지정
        # Feed Forward(오차 순전파)
        pred = model(x)
        loss = loss_fn(pred, y)
        tr_loss += loss
        # Backpropagation(오차 역전파)
        loss.backward()             # 역전파를 통해 각 파라미터에 대한 손실 기울기 계산
        optimizer.step()            # 옵티마이저가 모델의 파라미터를 업데이트
        optimizer.zero_grad()     # 옵티마이저의 기울기 값 초기화.
    tr_loss /= num_batches          # 모든 배치에서의 loss 평균
    return tr_loss.item()

* 검증 함수

In [ ]:
@torch.no_grad() # 데코레이터 다음 함수 실행시 기울기를 계산하지 않도록 설정
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()                      # 모델을 평가 모드로 설정
    x, y = x_val_tensor.to(device), y_val_tensor.to(device)
    pred = model(x)
    eval_loss = loss_fn(pred, y).item()    # 예측 값 pred와 실제 값 y 사이의 손실 계산
    return eval_loss, pred

[참조] 위 함수와 동일

In [ ]:
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()                      # 모델을 평가 모드로 설정
    with torch.no_grad():           # 평가 과정에서 기울기를 계산하지 않도록 설정
        x, y = x_val_tensor.to(device), y_val_tensor.to(device)
        pred = model(x)
        eval_loss = loss_fn(pred, y).item()    # 예측 값 pred와 실제 값 y 사이의 손실 계산
    return eval_loss, pred

#### 2) 학습

In [ ]:
epochs = 10
tr_loss_list, val_loss_list = [], []
for t in range(epochs):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    val_loss,_ = evaluate(x_val, y_val, model, loss_fn, device)
    tr_loss_list.append(tr_loss)
    val_loss_list.append(val_loss)
    print(f"Epoch {t+1}, train loss : {tr_loss:4f}, val loss : {val_loss:4f}")

#### 3) 학습결과 그래프

* 함수 생성

In [ ]:
def dl_learning_curve(tr_loss_list, val_loss_list):
    epochs = list(range(1, len(tr_loss_list)+1))
    plt.plot(epochs, tr_loss_list, label='train_err', marker = '.')
    plt.plot(epochs, val_loss_list, label='val_err', marker = '.')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
dl_learning_curve(tr_loss_list, val_loss_list)

### (3) 예측 및 평가

In [ ]:
_, pred = evaluate(x_test, y_test, model, loss_fn, device)
pred = pred.argmax(axis=1)

In [ ]:
print(pred.device, y_test.device)
print(type(pred), type(y_test))

In [ ]:
# cpu, numpy로 변환
pred = pred.cpu().numpy()

In [ ]:
cm = confusion_matrix(y_test.numpy(), pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
print(classification_report(y_test.numpy(), pred, digits = 4))

## 4.CNN 모델링2

### (1) 모델 설계

In [ ]:
model = nn.Sequential(
                    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2, stride=2),
                    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2, stride=2),
                    nn.Flatten(),
                    nn.Linear(64*7*7, 128),
                    nn.ReLU(),
                    nn.Linear(128, 10)
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

In [ ]:
summary(model, input_size=(1, 28, 28))

### (2) 학습

In [ ]:
epochs = 10
tr_loss_list, val_loss_list = [], []
for t in range(epochs):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    val_loss,_ = evaluate(x_val, y_val, model, loss_fn, device)
    tr_loss_list.append(tr_loss)
    val_loss_list.append(val_loss)
    print(f"Epoch {t+1}, train loss : {tr_loss:4f}, val loss : {val_loss:4f}")

In [ ]:
dl_learning_curve(tr_loss_list, val_loss_list)

### (3) 예측 및 평가

In [ ]:
_, pred = evaluate(x_test, y_test, model, loss_fn, device)
pred = pred.argmax(axis=1)

In [ ]:
# cpu, numpy로 변환
pred = pred.cpu().numpy()

In [ ]:
cm = confusion_matrix(y_test.numpy(), pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
print(classification_report(y_test.numpy(), pred, digits = 4))